# Case Metadata — one row per case, and the cached citation graph

The base table every other Case law notebook joins against, and the pass that builds the
shared graph cache. Primary key: `case_id`.

## Raw data
```
/project/jevans/Dawoon/Science of Science/Case law
├── metadata.csv           5,179,698 cases   id, jurisdiction, court, reporter, decision date
└── Edge_list.parquet     47,519,638 edges   (source, target) case ids
```

`source` **cites** `target`. That is measured, not assumed: over a 2M-edge sample joined to
decision years, 1,766,441 edges have the source newer than the target, 64,106 are same-year,
and **zero** have the source older. Every one of the 47.5M edges has both endpoints present in
`metadata.csv`, so nothing is dropped mapping ids into code space.

`metadata.csv` also has a `cites` column — it is a **reporter citation string** for the case
itself ("59 Mass. App. Dec. 120"), not a reference list. The references are the edge list.

## Output
`Case law/output/case_metadata.parquet`

| column | meaning |
|---|---|
| `case_id` | CAP case id |
| `decision_year` | year of `decision_date_original`, `<NA>` where unparseable |
| `decision_date` | the raw date string, kept so a partial date is still auditable |
| `jurisdiction`, `jurisdiction_id` | 61 jurisdictions |
| `court`, `court_id` | 3,221 courts |
| `reporter`, `reporter_id` | 413 reporters |
| `name_abbreviation` | short case name |
| `ref_count` | cases this case **cites** (out-degree in the edge list) |

## Also written: the graph cache
```
Case law/cache/case_graph.npz    c_from, c_to (code space), year, uni
Case law/cache/case_csr.npz      out_ptr/out_idx (cites), in_ptr/in_idx (cited by)
```
Case ids run 1–12,707,012 but only 5,179,698 exist, so `cl_common` maps them to dense codes —
the position in the sorted unique id array, the same device as `uni_mag` in the sibling
projects. **Run this notebook first**: every other one reads the cache.

## The year is the binding constraint

Every window metric (citation windows, CD windows, SB ages) needs both endpoints dated. The
year is taken as the leading four digits of `decision_date_original` rather than by a strict
date parse, because a partial date (`1897`, `1897-03`) still carries a usable year and a strict
parse discards it. Both counts are printed below so the difference is visible.

In [1]:
import os, sys, gc, time
import numpy as np, pandas as pd
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/Case law')
import cl_common as cl
OUT_FP = cl.out('case_metadata.parquet')
cl.preflight('case_metadata')
cl.summary()

case law : /project/jevans/Dawoon/Science of Science/Case law
output   : /project/jevans/Dawoon/Science of Science/Case law/output
cache    : /project/jevans/Dawoon/Science of Science/Case law/cache

  case_metadata               OK
  edges        0.20 GB  /project/jevans/Dawoon/Science of Science/Case law/Edge_list.parquet
  metadata     0.61 GB  /project/jevans/Dawoon/Science of Science/Case law/metadata.csv
  graph        0.44 GB  /project/jevans/Dawoon/Science of Science/Case law/cache/case_graph.npz
  csr          0.53 GB  /project/jevans/Dawoon/Science of Science/Case law/cache/case_csr.npz


## 1. Build the graph cache

`cl_common.build_csr()` reads both inputs once and writes `case_graph.npz` and `case_csr.npz`.
Idempotent — if the cache is already there this is a no-op, so re-running the notebook is free.

In [2]:
%%time
cl.build_csr()
out_ptr, out_idx, in_ptr, in_idx, year, uni = cl.load_csr()
n = len(uni)
print(f'cases {n:,}   edges {len(out_idx):,}')
print(f'with decision_year : {(year > 0).sum():,}  ({(year > 0).mean()*100:.1f}%)')
_y = year[year > 0]
print(f'year range {_y.min()}-{_y.max()}   median {int(np.median(_y))}')

CSR cache present: /project/jevans/Dawoon/Science of Science/Case law/cache/case_csr.npz
CSR cache present: /project/jevans/Dawoon/Science of Science/Case law/cache/case_csr.npz
cases 5,179,698   edges 47,519,638
with decision_year : 5,179,698  (100.0%)
year range 1666-2020   median 1977


## 2. Assemble and save

`ref_count` comes from the CSR out-degree rather than from a re-scan of the edge list, so the
column cannot disagree with the graph the other notebooks use.

In [3]:
%%time
m = cl.read_metadata()
code = np.searchsorted(uni, m['case_id'].to_numpy())
ref_count = (out_ptr[1:] - out_ptr[:-1]).astype(np.int32)

strict = pd.to_datetime(m['decision_date_original'], errors='coerce', format='%Y-%m-%d')
print(f"year from the leading 4 digits : {(m['decision_year'] > 0).sum():,} "
      f"({(m['decision_year'] > 0).mean()*100:.1f}%)")
print(f"year from a strict YYYY-MM-DD  : {strict.notna().sum():,} "
      f"({strict.notna().mean()*100:.1f}%)")
print(f"  recovered by the loose parse : {int((m['decision_year'] > 0).sum() - strict.notna().sum()):,}")
del strict

# Int32 with a real NA rather than a -1 sentinel: a sentinel year silently enters means.
_dy = pd.Series(m['decision_year'].to_numpy()).astype('Int32')
_dy[_dy <= 0] = pd.NA

meta = pd.DataFrame({
    'case_id':           m['case_id'].to_numpy(),
    'decision_year':     _dy,
    'decision_date':     m['decision_date_original'].to_numpy(),
    'jurisdiction':      m['jurisdiction__name'].to_numpy(),
    'jurisdiction_id':   pd.to_numeric(m['jurisdiction_id'], errors='coerce').astype('Int32'),
    'court':             m['court__name_abbreviation'].to_numpy(),
    'court_id':          pd.to_numeric(m['court_id'], errors='coerce').astype('Int32'),
    'reporter':          m['reporter__short_name'].to_numpy(),
    'reporter_id':       pd.to_numeric(m['reporter_id'], errors='coerce').astype('Int32'),
    'name_abbreviation': m['name_abbreviation'].to_numpy(),
    'ref_count':         ref_count[code],
}).sort_values('case_id').reset_index(drop=True)
del m; gc.collect()

meta.to_parquet(OUT_FP, index=False)
print(f'\nWROTE {OUT_FP}  ({len(meta):,} rows, {len(meta.columns)} cols, '
      f'{os.path.getsize(OUT_FP)/1e6:.0f} MB)')
print(f"  jurisdictions {meta.jurisdiction.nunique()}   courts {meta.court_id.nunique()}   "
      f"reporters {meta.reporter_id.nunique()}")
print(f"  ref_count mean {meta.ref_count.mean():.2f}  max {meta.ref_count.max():,}  "
      f"zero {int((meta.ref_count == 0).sum()):,}")
display(meta.head(5))
display(meta.groupby(meta.decision_year // 25 * 25).size().rename('cases').to_frame().tail(8))

[20s] metadata: 5,179,698 cases
year from the leading 4 digits : 5,179,698 (100.0%)
year from a strict YYYY-MM-DD  : 4,819,411 (93.0%)
  recovered by the loose parse : 360,287

WROTE /project/jevans/Dawoon/Science of Science/Case law/output/case_metadata.parquet  (5,179,698 rows, 11 cols, 152 MB)
  jurisdictions 61   courts 3221   reporters 413
  ref_count mean 9.17  max 2,951  zero 372,314


,case_id,decision_year,decision_date,jurisdiction,jurisdiction_id,court,court_id,reporter,reporter_id,name_abbreviation,ref_count
0,1,1976,1976-12-21,Mass.,4,Mass. App. Dec.,15176,Mass. App. Dec.,579,Burdette v. City of Boston,13
1,2,1976,1976-11-09,Mass.,4,Mass. App. Dec.,15176,Mass. App. Dec.,579,Merrimack Valley National Bank v. Baird,4
2,3,1977,1977-01-07,Mass.,4,Mass. App. Dec.,15176,Mass. App. Dec.,579,Schneiderman v. Commonwealth of Massachusetts,4
3,4,1976,1976-12-07,Mass.,4,Mass. App. Dec.,15176,Mass. App. Dec.,579,Levenson v. Bertolet,8
4,5,1976,1976-08-16,Mass.,4,Mass. App. Dec.,15176,Mass. App. Dec.,579,Commercial Plastics & Supply Corp. v. Ace Dash...,7


,cases
decision_year,
1825,63054
1850,141700
1875,368608
1900,562326
1925,593308
1950,720769
1975,1449154
2000,1259958
